# MIDAS — defense sweep on InternVL2.5-8B

Runs only the activation-steering evaluation. The activations and the fitted
defense are inherited from `ankitpokhrel/midas-chai-attack-on-internvl2-5-8b`, attached as a kernel source, so
nothing is recomputed.

**Session:** GPU T4 x2, Internet on.

The question: on the 2B the steering direction was decodable at 94.7% yet moving
along it changed nothing the model output. The 8B probe reaches 100%, so if the
feature is causal anywhere it should be here.

## 1. Check the hardware

Run this first. If it reports anything other than two T4s, stop and fix the
accelerator setting — everything below assumes this layout.

In [ ]:
import os
# Reduces fragmentation, which the first OOM on this hardware recommended.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, torch, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout)

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    major, minor = torch.cuda.get_device_capability(i)
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  gpu {i}: {torch.cuda.get_device_name(i)}  "
          f"sm_{major}{minor}  {total:.1f} GB")

# Gate on compute capability, as midas.models.internvl.resolve_dtype does.
# torch.cuda.is_bf16_supported() reports True on a T4 because recent PyTorch
# counts emulated bf16, but Turing has no bf16 tensor cores and the emulated
# path is slower than float16.
cc = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
torch_claim = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
bf16 = cc >= 8
fa2 = bf16
print(f"torch bf16 claim   : {torch_claim}  (counts emulation -- not what we use)")
print(f"native bfloat16    : {bf16}   -> dtype will be {'bfloat16' if bf16 else 'float16'}")
print(f"FlashAttention-2 ok: {fa2}   -> attention will run {'flash' if fa2 else 'eager'}")
if not fa2:
    print("  (expected on T4: sm_75 is Turing, FA2 needs sm_80+. Eager attention")
    print("   uses much more memory, so keep --max-tiles modest at first.)")

## 2. Install dependencies

Kaggle preinstalls PyTorch built against its own CUDA, so **do not reinstall
torch** — that risks breaking GPU support and wastes several minutes.

Two pins are load-bearing and cost us hours to diagnose locally:

- **`sentencepiece==0.2.1`.** Version 0.2.2 rejects InternVL's `tokenizer.model`
  as malformed even though its checksum matches the hub; `AutoTokenizer` swallows
  the error and returns `False`, which surfaces much later as `'bool' object has
  no attribute 'convert_tokens_to_ids'`. 0.1.99 also works but has no Python 3.13
  wheel, and Kaggle now runs 3.13 — pinning it makes pip try a source build,
  which fails and aborts the whole install.
- **No `numpy<2` pin.** Kaggle's preinstalled stack calls
  `numpy.dtypes.StringDType`, which only exists in numpy 2.0+. Forcing numpy 1.x
  makes `transformers` fail to import its model code with an error that names
  llama rather than numpy.
- **`transformers<4.50`.** InternVL2.5's remote code does not inherit from
  `GenerationMixin`, which 4.50 made mandatory. Kaggle ships 5.x, so this is a
  real downgrade and takes a minute.

The install is checked below rather than assumed: a failed pip run leaves the
preinstalled versions in place, and the notebook would then fail much later for
reasons that look unrelated.

In [ ]:
%pip install -q "transformers>=4.37,<4.50" "sentencepiece==0.2.1" protobuf \
    "accelerate>=0.26" timm einops opencv-python-headless scikit-learn pytest

import importlib, sentencepiece, transformers, accelerate
for mod in (transformers, sentencepiece, accelerate):
    importlib.reload(mod)

print("transformers :", transformers.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("accelerate   :", accelerate.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
assert (major, minor) < (4, 50), (
    f"transformers {transformers.__version__} is too new; InternVL2.5's remote "
    "code needs <4.50. The pip step above must have failed -- scroll up."
)
assert sentencepiece.__version__ == "0.2.1", (
    f"sentencepiece {sentencepiece.__version__} installed, need 0.2.1. "
    "0.2.2 cannot read InternVL's tokenizer and fails confusingly later."
)
print("version checks passed")

## 3. Load the code

The code and CHAI's released artefacts arrive as an attached Kaggle Dataset, not
a git clone, so this notebook needs no GitHub repository and no access token.
Upload it with `python scripts/kaggle_deploy.py upload`.

The dataset is read-only, so it is copied into the working directory before the
editable install.

In [ ]:
import os, shutil, subprocess, glob
from pathlib import Path

INPUT = Path("/kaggle/input")
print("contents of /kaggle/input:")
if INPUT.exists():
    for entry in sorted(INPUT.iterdir()):
        inner = sorted(p.name for p in entry.iterdir())[:12] if entry.is_dir() else []
        print(f"  {entry.name}/  -> {inner}")
else:
    print("  (missing entirely -- no dataset is attached to this kernel)")

# Search recursively: a dataset may mount its payload one level deeper than
# expected depending on how the archive was packed.
# Datasets mount under /kaggle/input/datasets/<user>/<slug>; kernel outputs
# mount under /kaggle/input/<kernel-slug>/. The sweep kernel attaches both, and
# the main run saved its own working copy of this source tree as output -- so an
# unqualified search finds that stale copy instead of the uploaded dataset.
candidates = sorted(
    glob.glob("/kaggle/input/**/pyproject.toml", recursive=True),
    key=lambda p: (0 if "/kaggle/input/datasets/" in p else 1, p),
)
assert candidates, (
    "MIDAS source not found under /kaggle/input. Attach the dataset via "
    "Add Data -> Your Datasets, or re-run scripts/kaggle_deploy.py upload. "
    "If the dataset was created moments before this kernel started, it may "
    "still have been processing -- wait for `kaggle datasets status` to read "
    "ready, then push again."
)
SOURCE = Path(candidates[0]).parent
# Not "/kaggle/working/midas": Kaggle puts /kaggle/working on sys.path, so a
# directory of that name shadows the installed package and `import midas`
# resolves to an empty namespace package with __file__ None.
WORK = Path("/kaggle/working/midas_src")
print("source dataset:", SOURCE)

# /kaggle/input is read-only and `pip install -e` needs to write egg-info.
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SOURCE, WORK)
os.chdir(WORK)

chai = WORK / "vendor" / "chai"
assert (chai / "training.json").exists(), (
    f"CHAI artefacts missing from the dataset (looked in {chai}); "
    "re-run the upload step"
)
frames = len(glob.glob(str(chai / "DriveLM_dataset/**/*.jpg"), recursive=True))
print(f"CHAI frames present: {frames} (expect 184)")
assert frames == 184, "frame count wrong -- the upload was incomplete"

%pip install -q -e .

# The editable install registers the console scripts (midas-eval and friends),
# which run as separate processes. It does not refresh this already-running
# kernel's import path, so point sys.path at the sources directly.
import importlib, sys, shutil as _shutil

sys.path.insert(0, str(WORK / "src"))
importlib.invalidate_caches()

import midas
assert midas.__file__, (
    "`midas` imported as a namespace package, so something on sys.path shadows "
    "it -- check for a directory of that name under /kaggle/working"
)
print("midas imported from", midas.__file__)

missing = [c for c in ("midas-eval", "midas-activations", "midas-probes")
           if _shutil.which(c) is None]
assert not missing, f"console scripts not on PATH: {missing}"
print("console scripts available")

## 4. Verify the data before spending GPU time

These checks are CPU-only and take seconds. They catch a bad clone before a
multi-hour run does.

## Inherit the fitted defense

The previous kernel's `/kaggle/working` is mounted under `/kaggle/input`.

In [ ]:
import glob, shutil
from pathlib import Path

npz = glob.glob("/kaggle/input/**/defense/train_8b.npz", recursive=True)
acts = glob.glob("/kaggle/input/**/activations/*.npz", recursive=True)
assert npz, "fitted defense not found; is the previous kernel attached as a source?"
print("defense artifact:", npz[0])
print("activation sweeps:", [Path(a).name for a in acts])

# Copy out of the read-only mount so the artifact's .json sits beside its .npz.
dest = Path("/kaggle/working/results/defense")
dest.mkdir(parents=True, exist_ok=True)
for ext in (".npz", ".json"):
    src = Path(npz[0]).with_suffix(ext)
    shutil.copy2(src, dest / src.name)

from midas.steering import DefenseArtifact
art = DefenseArtifact.load(dest / "train_8b")
print("")
print(f"steering layers: {art.layer_indices}")
print(f"gate layer     : {art.gate_layer}")

## Run the sweep

One and five layers, four strengths. `clean_keep` is the utility measure: the
share of unattacked frames where the defended model still prefers what the
undefended model preferred.

This run sweeps the **step cap** as well. The previous sweep showed the
intervention firing (136,725 activations modified) and moving the decision
margin by 1.45 nats, yet not flipping a single scene — but at strength 10 about
half the steps hit the 25%-of-norm safety cap, so the realised push was smaller
than requested. Raising the cap separates two very different conclusions:

* the direction is **inert** — the model's decision does not depend on it; or
* it needs a push **larger than is safe** — which is a different claim, and a
  weaker defense.

The cap exists because an unbounded step once reduced the model to word salad,
so `clean_keep` is the thing to watch: if it falls, the cap was load-bearing.

In [ ]:
!midas-sweep-defense /kaggle/working/results/defense/train_8b \
    --model OpenGVLab/InternVL2_5-8B --device-map auto --max-tiles 6 \
    --strengths 0 5 10 --top-layers 5 --max-step-ratio 0.25 0.5 1.0 \
    --out /kaggle/working/results/defense/sweep_8b.json

In [ ]:
import json
from pathlib import Path

path = Path("/kaggle/working/results/defense/sweep_8b.json")
assert path.exists(), "the sweep produced no output -- check the cell above"
rows = json.loads(path.read_text())

base = next(r["attack_rate"] for r in rows if r["strength"] == 0)
print(f"{'configuration':24s} {'attack':>8s} {'vs none':>9s} {'utility':>9s}")
for r in rows:
    label = "no defense" if r["strength"] == 0 else (
        f"{r.get('top_layers', 1)} layer(s) s={r['strength']:g}"
    )
    delta = "--" if r["strength"] == 0 else f"{r['attack_rate'] - base:+.1%}"
    print(f"{label:24s} {r['attack_rate']:8.1%} {delta:>9s} {r['clean_keep']:8.1%}")

n = rows[0]["n"]
spread = max(r["attack_rate"] for r in rows) - min(r["attack_rate"] for r in rows)
print("")
print(f"spread: {spread:.1%} = {round(spread * n)} scenes of {n}")
print("At n=15 the 95% interval is about +/-25 points, so a spread of one or two")
print("scenes is indistinguishable from no effect.")